In [1]:
"""
End-to-End Learning of Communications Systems Without a Channel Model
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np


def awgn_channel(x: torch.Tensor, snr_db: float) -> torch.Tensor:
    sigma2 = 10 ** (-snr_db / 10.0)
    noise = torch.randn_like(x) * math.sqrt(sigma2 / 2)  # /2 for real+imag
    return x + noise


def rbf_channel(x: torch.Tensor, snr_db: float):

    batch = x.shape[0]
    N_real = x.shape[1]          # 2N reals (real & imag interleaved: [re_0,im_0,...])
    N = N_real // 2

    #h = h_re + j*h_im
    h_re = torch.randn(batch, 1) / math.sqrt(2)
    h_im = torch.randn(batch, 1) / math.sqrt(2)

    # (h_re + j*h_im)(x_re + j*x_im)
    x_re = x[:, 0::2]  # even indices → real parts  (batch, N)
    x_im = x[:, 1::2]  # odd  indices → imag parts  (batch, N)

    y_re = h_re * x_re - h_im * x_im
    y_im = h_re * x_im + h_im * x_re

    # Interleave back
    y = torch.zeros_like(x)
    y[:, 0::2] = y_re
    y[:, 1::2] = y_im

  
    sigma2 = 10 ** (-snr_db / 10.0)
    noise = torch.randn_like(y) * math.sqrt(sigma2 / 2)
    y = y + noise

    
    h = torch.cat([h_re.expand(batch, N), h_im.expand(batch, N)], dim=1)  # (batch, 2N)
    return y, h



class Transmitter(nn.Module):
    """
      Embedding(M, M)  →  ELU  →  Dense(2N, linear)  →  normalize
    """

    def __init__(self, M: int, N: int):
        super().__init__()
        self.M = M
        self.N = N
        
        self.embedding = nn.Embedding(M, M)
        self.hidden    = nn.Linear(M, M)
        self.output    = nn.Linear(M, 2 * N)   # 2N reals = N complex symbols

    def forward(self, m: torch.Tensor) -> torch.Tensor:
        """
        m : (batch,) integer message indices
        returns x : (batch, 2N) normalised real representation
        """
        e = F.elu(self.embedding(m))           # (batch, M)
        h = F.elu(self.hidden(e))              # (batch, M)
        x = self.output(h)                     # (batch, 2N)  linear activations


        avg_energy = (x ** 2).mean()          
        x = x / (x.norm(dim=1, keepdim=True) / math.sqrt(self.N) + 1e-8)
        return x




class ReceiverAWGN(nn.Module):
    """
      C2R layer (= treat 2N reals as input)
      → Dense(M, ReLU)
      → Dense(M, softmax)
    """

    def __init__(self, M: int, N: int):
        super().__init__()
        self.fc1 = nn.Linear(2 * N, M)
        self.fc2 = nn.Linear(M, M)

    def forward(self, y: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.fc1(y))
        p = F.softmax(self.fc2(h), dim=1)
        return p


class ReceiverRBF(nn.Module):
    """
    
      C2R
      → Dense(20, tanh)  ← estimates channel h
      → Dense(2,  linear) ← produces complex h estimate
      → divide y by h_hat (channel equalisation)
      → Dense(M, ReLU)
      → Dense(M, softmax)

    """

    def __init__(self, M: int, N: int):
        super().__init__()
        self.N = N
        
        self.ce1 = nn.Linear(2 * N, 20)
        self.ce2 = nn.Linear(20, 2)       # outputs (h_re_est, h_im_est)
        # Detection sub-network (same as AWGN receiver)
        self.fc1  = nn.Linear(2 * N, M)
        self.fc2  = nn.Linear(M, M)

    def forward(self, y: torch.Tensor) -> torch.Tensor:
        """y : (batch, 2N)  →  p : (batch, M)"""
        batch = y.shape[0]
        N = self.N

        
        h_est = torch.tanh(self.ce1(y))       # (batch, 20)
        h_est = self.ce2(h_est)               # (batch, 2): [h_re, h_im]
        h_re  = h_est[:, 0:1]                 # (batch, 1)
        h_im  = h_est[:, 1:2]                 # (batch, 1)

        y_re = y[:, 0::2]   # (batch, N)
        y_im = y[:, 1::2]   # (batch, N)
        denom = h_re ** 2 + h_im ** 2 + 1e-8   # (batch, 1)
        eq_re = ( h_re * y_re + h_im * y_im) / denom
        eq_im = (-h_im * y_re + h_re * y_im) / denom

        y_eq = torch.zeros(batch, 2 * N, device=y.device)
        y_eq[:, 0::2] = eq_re
        y_eq[:, 1::2] = eq_im

        
        h = F.relu(self.fc1(y_eq))
        p = F.softmax(self.fc2(h), dim=1)
        return p



def sample_policy(x: torch.Tensor, sigma_pi: float) -> torch.Tensor:
    """
    Sample a perturbed action x_p from the Gaussian policy.
    x        : (batch, 2N)  transmitter output (clean)
    sigma_pi : exploration std (fixed, not learned)
    returns x_p : (batch, 2N)  noisy transmitted signal
    """
    scale = math.sqrt(1 - sigma_pi ** 2)
    w = torch.randn_like(x) * sigma_pi
    x_p = scale * x + w
    return x_p


def log_policy(x_p: torch.Tensor, x: torch.Tensor, sigma_pi: float) -> torch.Tensor:
    """
    Compute log π(x_p | x) for each example in the batch.
    Returns : (batch,)  log-probability of each sample under the policy.
    This is what we differentiate w.r.t. θ_T to get the policy gradient.
    """
    scale = math.sqrt(1 - sigma_pi ** 2)
    diff  = x_p - scale * x                        # (batch, 2N)

    log_p = -(diff ** 2).sum(dim=1) / (sigma_pi ** 2)   # (batch,)
    return log_p




def train_receiver(
    transmitter: Transmitter,
    receiver: nn.Module,
    opt_rx: torch.optim.Optimizer,
    channel_fn,            # callable: x → y (and optionally h)
    M: int,
    batch_size: int,
    n_steps: int,
    channel_type: str,
):
    """
    Runs n_steps gradient descent steps on the receiver.
    """
    transmitter.eval()    # transmitter frozen
    receiver.train()

    total_loss = 0.0
    for _ in range(n_steps):
        # 1. Draw random messages
        m = torch.randint(0, M, (batch_size,))

        # 2. Transmitter encodes 
        with torch.no_grad():
            x = transmitter(m)

        # 3. Pass through channel
        if channel_type == "awgn":
            y = channel_fn(x)
        else:   # rbf: channel returns (y, h) — receiver only uses y
            y, _ = channel_fn(x)

        # 4. Receiver predicts distribution over M
        p = receiver(y)    # (batch, M)

        # 5. Cross-entropy loss  
        loss = F.cross_entropy(torch.log(p + 1e-9), m)

        # 6. Gradient step on receiver parameters
        opt_rx.zero_grad()
        loss.backward()
        opt_rx.step()

        total_loss += loss.item()

    return total_loss / n_steps



def train_transmitter(
    transmitter: Transmitter,
    receiver: nn.Module,
    opt_tx: torch.optim.Optimizer,
    channel_fn,
    M: int,
    batch_size: int,
    n_steps: int,
    sigma_pi: float,
    channel_type: str,
):
    """
    Runs n_steps RL policy-gradient steps on the transmitter.
    """
    transmitter.train()
    receiver.eval()    # receiver frozen

    total_loss = 0.0
    for _ in range(n_steps):
        # 1. Draw random messages
        m = torch.randint(0, M, (batch_size,))

        # 2. Transmitter encodes — keep computation graph for θ_T
        x = transmitter(m)   # (batch, 2N), requires_grad through θ_T

        # 3. Sample stochastic policy (perturb x)
        x_p = sample_policy(x.detach(), sigma_pi)   # no graph through x_p itself

        # 4. Channel (black box — no grad)
        with torch.no_grad():
            if channel_type == "awgn":
                y = channel_fn(x_p)
            else:
                y, _ = channel_fn(x_p)

        # 5. Receiver produces per-example losses (black box)
        with torch.no_grad():
            p = receiver(y)   # (batch, M)
            # Per-example CE loss: l^(i) = -log p[m^(i)]
            l = -torch.log(p[torch.arange(batch_size), m] + 1e-9)   # (batch,)

        # 6. Compute log π(x_p | x)  — this is differentiable w.r.t. θ_T
        #    because x was produced by the transmitter graph.
        log_p = log_policy(x_p, x, sigma_pi)   # (batch,)

        # 7. Policy gradient loss  
    
        pg_loss = (l.detach() * log_p).mean()

        # 8. Gradient step on transmitter parameters
        opt_tx.zero_grad()
        pg_loss.backward()
        opt_tx.step()

        total_loss += l.mean().item()

    return total_loss / n_steps



def alternating_training(
    M: int = 256,
    N: int = 4,
    channel_type: str = "awgn",   # "awgn" or "rbf"
    snr_db: float = 10.0,
    n_iterations: int = 500,
    rx_steps_per_iter: int = 50,
    tx_steps_per_iter: int = 50,
    batch_rx: int = 256,
    batch_tx: int = 256,
    sigma_pi: float = math.sqrt(0.02),
    lr: float = 1e-3,
):
    """
    Full alternating training algorithm.
    Returns lists of error rates recorded at each iteration.
    """

    
    tx = Transmitter(M, N)
    rx = ReceiverAWGN(M, N) if channel_type == "awgn" else ReceiverRBF(M, N)

    # Adam optimiser 
    opt_tx = torch.optim.Adam(tx.parameters(), lr=lr)
    opt_rx = torch.optim.Adam(rx.parameters(), lr=lr)

    
    if channel_type == "awgn":
        channel_fn = lambda x: awgn_channel(x, snr_db)
    else:
        channel_fn = lambda x: rbf_channel(x, snr_db)

    
    error_rates = []

    for iteration in range(n_iterations):

        # Phase (i): train receiver
        train_receiver(
            tx, rx, opt_rx, channel_fn,
            M, batch_rx, rx_steps_per_iter, channel_type
        )

        # Phase (ii): train transmitter
        train_transmitter(
            tx, rx, opt_tx, channel_fn,
            M, batch_tx, tx_steps_per_iter, sigma_pi, channel_type
        )

        # error rate 
        er = evaluate(tx, rx, channel_fn, M, n_samples=2000, channel_type=channel_type)
        error_rates.append(er)

        if (iteration + 1) % 50 == 0:
            print(f"  [{channel_type.upper()}] iter {iteration+1:4d}/{n_iterations} | "
                  f"error rate = {er:.4f}")

    return error_rates




@torch.no_grad()
def evaluate(tx, rx, channel_fn, M, n_samples=4096, channel_type="awgn"):
    """Compute block error rate (fraction of messages decoded wrongly)."""
    tx.eval(); rx.eval()
    m      = torch.randint(0, M, (n_samples,))
    x      = tx(m)

    if channel_type == "awgn":
        y  = channel_fn(x)
    else:
        y, _ = channel_fn(x)

    p      = rx(y)
    m_hat  = p.argmax(dim=1)
    error_rate = (m_hat != m).float().mean().item()
    tx.train(); rx.train()
    return error_rate


@torch.no_grad()
def evaluate_over_snr(tx, rx, channel_fn_factory, M, N,
                      snr_range, channel_type="awgn", n_samples=4000):
    """Sweep over SNR values and record error rates."""
    ers = []
    for snr_db in snr_range:
        cf = channel_fn_factory(snr_db)
        er = evaluate(tx, rx, cf, M, n_samples, channel_type)
        ers.append(er)
    return ers




if __name__ == "__main__":
    torch.manual_seed(42)

    
    M = 256       
    N = 4         
    N_ITER = 500  

    print("=" * 60)
    print("Training on AWGN channel  (SNR = 10 dB)")
    print("=" * 60)
    er_awgn = alternating_training(
        M=M, N=N,
        channel_type="awgn",
        snr_db=10.0,
        n_iterations=N_ITER,
        rx_steps_per_iter=10,   
        tx_steps_per_iter=10,
        batch_rx=256,
        batch_tx=256,
    )

    print()
    print("=" * 60)
    print("Training on RBF channel  (SNR = 20 dB)")
    print("=" * 60)
    er_rbf = alternating_training(
        M=M, N=N,
        channel_type="rbf",
        snr_db=20.0,
        n_iterations=N_ITER,
        rx_steps_per_iter=10,
        tx_steps_per_iter=10,
        batch_rx=256,
        batch_tx=256,
    )

    # Plot 1: convergence curves  
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(er_awgn, label="AWGN – Alternating training", color="#185FA5")
    ax.plot(er_rbf,  label="RBF  – Alternating training",  color="#0F6E56", linestyle="--")
    ax.set_xlabel("Iterations")
    ax.set_ylabel("Error Rate")
    ax.set_title("Convergence of alternating training (Algorithm 1)")
    ax.legend()
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("./convergence.png", dpi=150)
    plt.close()
    print("\nSaved convergence.png")

    


   

Training on AWGN channel  (SNR = 10 dB)
  [AWGN] iter   50/500 | error rate = 0.0090
  [AWGN] iter  100/500 | error rate = 0.0070
  [AWGN] iter  150/500 | error rate = 0.0080
  [AWGN] iter  200/500 | error rate = 0.0105
  [AWGN] iter  250/500 | error rate = 0.0060
  [AWGN] iter  300/500 | error rate = 0.0085
  [AWGN] iter  350/500 | error rate = 0.0060
  [AWGN] iter  400/500 | error rate = 0.0125
  [AWGN] iter  450/500 | error rate = 0.0090
  [AWGN] iter  500/500 | error rate = 0.0115

Training on RBF channel  (SNR = 20 dB)
  [RBF] iter   50/500 | error rate = 0.1400
  [RBF] iter  100/500 | error rate = 0.0845
  [RBF] iter  150/500 | error rate = 0.0960
  [RBF] iter  200/500 | error rate = 0.0835
  [RBF] iter  250/500 | error rate = 0.0785
  [RBF] iter  300/500 | error rate = 0.0630
  [RBF] iter  350/500 | error rate = 0.0625
  [RBF] iter  400/500 | error rate = 0.0715
  [RBF] iter  450/500 | error rate = 0.0665
  [RBF] iter  500/500 | error rate = 0.0685

Saved convergence.png
